# AI Video Studio — Kaggle GPU Worker

Run the eight phases in order. Select **GPU T4 x2** and enable **Internet**.
The GitHub repository must contain `backend/`, `site/`, and `requirements-kaggle.txt` at its root.

**Enabled model:** SVD-XT only. CogVideoX, Wan 2.1 and LTX-Video are not implemented in this release.


In [ ]:
# PHASE 1 — Clone and validate the project repository
from pathlib import Path
import subprocess, sys, shutil

REPO_URL = "https://github.com/suryaprakash8t/suryaprakash8t.github.io.git"
PROJECT = Path("/kaggle/working/AI_Video_Studio")

if not (PROJECT / ".git").exists():
    if PROJECT.exists():
        shutil.rmtree(PROJECT)
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=True)

# Recover automatically if the project files were pushed one folder too deep.
nested = PROJECT / "AI_Video_Studio"
if not (PROJECT / "backend" / "app.py").is_file() and (nested / "backend" / "app.py").is_file():
    shutil.copytree(nested, PROJECT, dirs_exist_ok=True)

required = [
    PROJECT / "backend" / "app.py",
    PROJECT / "backend" / "runtime.py",
    PROJECT / "requirements-kaggle.txt",
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Project structure is incomplete. Missing: " + ", ".join(missing))

sys.path.insert(0, str(PROJECT))
print("Project:", PROJECT)
print("[PASS] backend/app.py")
print("[PASS] backend/runtime.py")
print("[PASS] requirements-kaggle.txt")
print("Website present:", (PROJECT / "site" / "index.html").is_file())
print("[PASS] PHASE 1")


In [ ]:
# PHASE 2 — Install project dependencies and system tools
import subprocess, sys, shutil, platform
from pathlib import Path

BASE = Path("/kaggle/working/AI_Video_Studio")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(BASE / "requirements-kaggle.txt")],
    check=True
)

# FFmpeg is required by the video editor.
subprocess.run(
    ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq ffmpeg curl"],
    check=True
)

# Install cloudflared only when it is not already available.
if not shutil.which("cloudflared"):
    machine = platform.machine().lower()
    asset = "cloudflared-linux-arm64" if machine in ("aarch64", "arm64") else "cloudflared-linux-amd64"
    url = f"https://github.com/cloudflare/cloudflared/releases/latest/download/{asset}"
    subprocess.run(
        ["bash", "-lc", f"curl -L --fail --retry 3 -o /usr/local/bin/cloudflared '{url}' && chmod +x /usr/local/bin/cloudflared"],
        check=True
    )

print("Python:", sys.version.split()[0])
print("FFmpeg:", shutil.which("ffmpeg"))
print("cloudflared:", shutil.which("cloudflared"))
subprocess.run(["cloudflared", "--version"], check=True)
print("[PASS] PHASE 2")


In [ ]:
# PHASE 3 — Validate project files and Python import path
from pathlib import Path
import sys

BASE = Path("/kaggle/working/AI_Video_Studio")
required = [
    BASE / "backend" / "app.py",
    BASE / "backend" / "runtime.py",
    BASE / "backend" / "__init__.py",
    BASE / "requirements-kaggle.txt",
    BASE / "site" / "index.html",
    BASE / "site" / "tunnel-config.json",
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing project files: " + ", ".join(missing))

if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

for path in required:
    print("[PASS]", path.relative_to(BASE))
print("[PASS] PHASE 3")


In [ ]:
# PHASE 4 — Compile backend and validate installed dependencies
import py_compile, torch, diffusers, fastapi, uvicorn
from pathlib import Path

BASE = Path("/kaggle/working/AI_Video_Studio")
for relative in ("backend/runtime.py", "backend/app.py"):
    py_compile.compile(str(BASE / relative), doraise=True)
    print("[PASS]", relative)

print("PyTorch:", torch.__version__)
print("Diffusers:", diffusers.__version__)
print("FastAPI:", fastapi.__version__)
print("CUDA available:", torch.cuda.is_available())
print("[PASS] PHASE 4")


In [ ]:
# PHASE 5 — Check GPU and load SVD-XT
import torch, psutil, shutil
from backend import runtime

print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU is unavailable. In Kaggle Settings select GPU T4 x2.")

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} | {props.total_memory / 1024**3:.2f} GB VRAM")

print("RAM:", round(psutil.virtual_memory().total / 1024**3, 2), "GB")
print("Disk free:", round(shutil.disk_usage("/kaggle/working").free / 1024**3, 2), "GB")

# HF_TOKEN is optional for public model access. Never print the token.
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("HF_TOKEN: available (value hidden)")
else:
    print("HF_TOKEN: not configured; public model access will be attempted")

cache_dir = "/kaggle/working/AI_Video_Studio/models"
print("Loading SVD-XT. The first download can take a long time.")
runtime.load_svd(cache_dir)
if runtime.PIPELINE is None:
    raise RuntimeError("SVD-XT did not load.")
print("Model loaded:", True)
print("[PASS] PHASE 5")


In [ ]:
# PHASE 6 — Start the authenticated FastAPI backend
import os, threading, time, urllib.request, importlib
import uvicorn
from kaggle_secrets import UserSecretsClient

try:
    API_KEY = UserSecretsClient().get_secret("AI_VIDEO_API_KEY")
except Exception as exc:
    raise RuntimeError("Kaggle secret AI_VIDEO_API_KEY is unavailable. Add it in Add-ons > Secrets and enable notebook access.") from exc

if not API_KEY:
    raise RuntimeError("AI_VIDEO_API_KEY is empty. Add the secret in Kaggle and enable notebook access.")

os.environ["AI_STUDIO_API_KEY"] = API_KEY
backend_app = importlib.import_module("backend.app")
backend_app.API_KEY = API_KEY
app = backend_app.app

def _local_health():
    with urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=4) as response:
        return response.read().decode()

# Reuse an already-running backend when this cell is rerun.
try:
    health_text = _local_health()
    print("Existing backend reused:", health_text)
except Exception:
    def _serve_ai_video_studio():
        uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

    API_SERVER_THREAD = threading.Thread(target=_serve_ai_video_studio, daemon=True)
    API_SERVER_THREAD.start()

    for attempt in range(60):
        try:
            health_text = _local_health()
            print("Health:", health_text)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("FastAPI did not start on port 8000. Check the cell output above.")

print("[PASS] PHASE 6 — Backend running on port 8000")


In [ ]:
# PHASE 7 — Start/reuse Cloudflare Quick Tunnel and update GitHub Pages config
import subprocess, time, re, pathlib, json, base64, urllib.request, urllib.error
import psutil
from kaggle_secrets import UserSecretsClient

LOCAL_HEALTH = "http://127.0.0.1:8000/api/health"
with urllib.request.urlopen(LOCAL_HEALTH, timeout=10) as response:
    if response.status != 200:
        raise RuntimeError("Local backend health check failed before starting Cloudflare.")

log_path = pathlib.Path("/kaggle/working/cloudflared.log")
url_pattern = r"https://[a-zA-Z0-9-]+\.trycloudflare\.com"

def _cloudflared_processes():
    found = []
    for proc in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            name = (proc.info.get("name") or "").lower()
            cmd = " ".join(proc.info.get("cmdline") or [])
            if "cloudflared" in name and "tunnel" in cmd and "127.0.0.1:8000" in cmd:
                found.append(proc)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return found

def _public_health(url):
    with urllib.request.urlopen(url + "/api/health", timeout=15) as response:
        data = json.loads(response.read().decode())
        return response.status == 200 and data.get("status") == "ok"

tunnel_url = None
existing = _cloudflared_processes()

# Reuse a live tunnel from an earlier execution instead of requesting another Quick Tunnel.
if existing and log_path.exists():
    match = re.search(url_pattern, log_path.read_text(errors="ignore"))
    if match:
        candidate = match.group(0)
        try:
            if _public_health(candidate):
                tunnel_url = candidate
                print("Reusing existing tunnel:", tunnel_url)
        except Exception:
            pass

if not tunnel_url:
    # Stop only stale cloudflared processes associated with this Kaggle backend.
    for proc in existing:
        try:
            proc.terminate()
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    time.sleep(2)
    log_path.write_text("", encoding="utf-8")
    log_handle = open(log_path, "w", encoding="utf-8")
    CLOUDFLARED_PROCESS = subprocess.Popen(
        ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
        text=True,
    )

    for _ in range(90):
        log_text = log_path.read_text(errors="ignore") if log_path.exists() else ""
        match = re.search(url_pattern, log_text)
        if match:
            tunnel_url = match.group(0)
            break
        if "status 429" in log_text or "Tunnelstore request failed" in log_text:
            raise RuntimeError(
                "Cloudflare Quick Tunnel returned HTTP 429 (rate limited). "
                "Do not rerun this cell repeatedly. Keep Kaggle running and wait before one retry."
            )
        if CLOUDFLARED_PROCESS.poll() is not None:
            raise RuntimeError("cloudflared exited before providing a URL. Log:\\n" + log_text[-3000:])
        time.sleep(1)

    if not tunnel_url:
        log_text = log_path.read_text(errors="ignore") if log_path.exists() else "No Cloudflare log."
        raise RuntimeError("Cloudflare URL was not detected. Log:\\n" + log_text[-3000:])

    # Confirm the public hostname reaches the local FastAPI backend.
    public_ok = False
    last_error = None
    for _ in range(12):
        try:
            if _public_health(tunnel_url):
                public_ok = True
                break
        except Exception as exc:
            last_error = exc
        time.sleep(3)
    if not public_ok:
        raise RuntimeError(f"Tunnel URL was created but public health check failed: {last_error}")

print("Tunnel:", tunnel_url)

# Save only the public tunnel URL in the repository. Never upload API keys.
try:
    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    raise RuntimeError("GITHUB_TOKEN is unavailable. Add it in Kaggle Secrets with repository Contents read/write access.") from exc

if not github_token:
    raise RuntimeError("GITHUB_TOKEN is empty. Add it in Kaggle Secrets with repository Contents read/write access.")

owner, repo, file_path = "suryaprakash8t", "suryaprakash8t.github.io", "site/tunnel-config.json"
api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{file_path}"
headers = {
    "Authorization": f"Bearer {github_token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
file_sha = None
get_req = urllib.request.Request(api_url, headers=headers, method="GET")
try:
    with urllib.request.urlopen(get_req, timeout=20) as response:
        file_sha = json.loads(response.read().decode()).get("sha")
except urllib.error.HTTPError as exc:
    if exc.code != 404:
        detail = exc.read().decode(errors="ignore")
        raise RuntimeError(f"GitHub could not read site/tunnel-config.json (HTTP {exc.code}): {detail}") from exc

encoded = base64.b64encode(json.dumps({"api_url": tunnel_url}, indent=2).encode()).decode()
payload = {
    "message": "Update Kaggle Cloudflare tunnel URL",
    "content": encoded,
    "branch": "main",
}
if file_sha:
    payload["sha"] = file_sha

put_req = urllib.request.Request(
    api_url,
    data=json.dumps(payload).encode(),
    headers=headers,
    method="PUT",
)
try:
    with urllib.request.urlopen(put_req, timeout=30) as response:
        result = json.loads(response.read().decode())
except urllib.error.HTTPError as exc:
    detail = exc.read().decode(errors="ignore")
    if exc.code == 403:
        raise RuntimeError(
            "GitHub returned HTTP 403. Check that GITHUB_TOKEN is a fine-grained token "
            "for this repository with Contents: Read and write. GitHub response: " + detail
        ) from exc
    raise RuntimeError(f"GitHub tunnel-config update failed (HTTP {exc.code}): {detail}") from exc

print("GitHub tunnel config updated.")
print("Commit:", result["commit"]["sha"][:12])
print("[PASS] PHASE 7 — Keep this Kaggle session running.")


In [ ]:
# PHASE 8 — Final local and public API smoke tests
import urllib.request, json

for route in ("/api/health", "/api/models", "/api/system"):
    with urllib.request.urlopen("http://127.0.0.1:8000" + route, timeout=15) as response:
        payload = json.loads(response.read().decode())
        if response.status != 200:
            raise RuntimeError(f"{route} returned HTTP {response.status}")
        print(route, "OK")

health = json.loads(urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=10).read().decode())
if health.get("status") != "ok" or health.get("model_loaded") is not True:
    raise RuntimeError("Backend health is not ready: " + str(health))

print("[PASS] PHASE 8 — Backend checks passed.")


## Daily startup and known limits

After a Kaggle session restarts, run Phases 1–8 in order. Model weights may need to download again. Keep the Kaggle session running while using the website.

- **Only SVD-XT is implemented and enabled** in this release. CogVideoX, Wan 2.1 and LTX-Video are planned, not working models.
- SVD-XT generates 25 frames and does not follow text prompts. The prompt field is retained as a note only.
- Generation is capped at 5–10 inference steps by the backend.
- Kaggle Quick Tunnel URLs are temporary and can change after a session restart. Phase 7 reuses a live tunnel when possible and updates `site/tunnel-config.json`.
- Cloudflare may return HTTP 429 when too many Quick Tunnels are requested. This is an external rate limit; the notebook reports it and does not loop retries.
- Never put `AI_VIDEO_API_KEY`, `HF_TOKEN`, or `GITHUB_TOKEN` in the public website files.
- Download generated videos before stopping Kaggle. Kaggle storage and sessions are temporary.
- Video joining is most reliable when clips have matching codecs, frame rates and dimensions.
